# StormBox: storm risk models, pressure vs wind (Colab)

Version 2 of the storm risk notebook. It trains the same kind of model three ways and compares them **on the same hours**:

| Model set | Uses | When the box uses it |
|---|---|---|
| `pressure` | barometer only (same as version 1) | live, if the barometer works |
| `wind` | anemometer only (wind speed + gusts) | **live, while the barometer is down** |
| `both` | pressure + wind | storm **replay** (the replay files contain both) |

**The question each model answers:** *Will a tropical storm or hurricane center pass within 150 km of here in the next 12 hours?*

**Data:** the same NOAA buoy/coastal-station files as version 1. They already contain wind speed (`WSPD`) and gusts (`GST`), so nothing new to find.

**How to run:** `Runtime → Run all` (about 5–15 minutes the first time). A zip downloads at the end.

**Honest limits of the wind model (say these if judges ask):**
- The anemometer must be **outside, in the open**. Indoors it reads zero.
- NOAA wind is measured over open water or at exposed coastal stations, which is windier than a backyard. Changes over time transfer better than exact numbers, so the box may be a little less sensitive than the test numbers show.
- The live model needs ~6 hours of history before it can score. The **demo uses replay**, which has real NOAA wind and pressure.

In [ ]:
# STEP 0: imports (all already installed in Colab)
import os, io, re, gzip, json, math, time, shutil, urllib.request, urllib.error
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib, sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

for d in ["data/ndbc", "out/models", "out/replay", "out/figures"]:
    os.makedirs(d, exist_ok=True)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 20)
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

## Step 1: settings

Same stations, years, and label as version 1. New: wind limits and which model set the box uses live.

In [ ]:
# STEP 1: settings
STATIONS = [
    # Gulf side buoys
    "42036", "42039", "42003", "42013", "42022", "42023",
    # Atlantic side buoys
    "41009", "41010",
    # coastal stations (C-MAN / NOS)
    "venf1", "cdrf1", "sauf1", "lkwf1", "kywf1", "smkf1", "fwyf1", "mlrf1",
    "apcf1", "pacf1", "npsf1", "vcaf1", "cwbf1",
]
YEARS       = range(2005, 2026)
TRAIN_YEARS = range(2005, 2019)
VAL_YEARS   = range(2019, 2022)   # used only to choose the alarm threshold
TEST_YEARS  = range(2022, 2026)   # storms the models never saw

RADIUS_KM   = 150     # "storm close" = center within this distance
HORIZON_H   = 12      # ...within the next 12 hours
NORMAL_KM   = 500     # "normal weather" = no storm within 500 km for +/- 48 h
SEASON_MONTHS = range(6, 12)       # June-November
MAX_DEPTH   = 3       # small tree = easy to draw and explain
MIN_LEAF    = 100
FA_PER_WEEK_TARGET = 1.0           # at most 1 false alarm per week of normal weather

P_MIN, P_MAX = 850.0, 1090.0       # pressure outside this = bad reading (hPa)
W_MAX, G_MAX = 99.0, 99.0          # NOAA writes 99.0 for missing wind/gust, so only values below 99 m/s are kept

LIVE_SET = "wind"                  # what the box runs live while the barometer is down

HURDAT_FALLBACK = "https://www.nhc.noaa.gov/data/hurdat/hurdat2-1851-2024-040425.txt"

## Step 2: download helpers + station locations

In [ ]:
# STEP 2: download helper + station coordinates
def fetch(url, retries=2, timeout=60):
    # Returns the file bytes, or None if the file does not exist (404).
    err = None
    for attempt in range(retries + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "StormBox-student-project"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            err = e
        except Exception as e:
            err = e
        time.sleep(1 + attempt)
    print("  ! failed:", url, err)
    return None

def load_station_coords():
    raw = fetch("https://www.ndbc.noaa.gov/data/stations/station_table.txt")
    if raw is None:
        raise RuntimeError("Could not download the NDBC station table")
    coords = {}
    for line in raw.decode("latin-1").splitlines():
        if line.startswith("#"):
            continue
        parts = line.split("|")
        if len(parts) < 7:
            continue
        m = re.match(r"\s*([\d.]+)\s+([NS])\s+([\d.]+)\s+([EW])", parts[6])
        if m:
            lat = float(m[1]) * (1 if m[2] == "N" else -1)
            lon = float(m[3]) * (1 if m[4] == "E" else -1)
            coords[parts[0].strip().lower()] = (lat, lon)
    return coords

COORDS = load_station_coords()
STATION_POS = {s.lower(): COORDS[s.lower()] for s in STATIONS if s.lower() in COORDS}
print(f"{len(STATION_POS)} of {len(STATIONS)} stations found in the NOAA table")

## Step 3: download pressure + wind history (NDBC)

From each 10-minute row we keep three numbers:
- `p`: pressure (hPa, sea level). Column `PRES` (newer files) or `BAR` (older files).
- `w`: wind speed (m/s, averaged over a few minutes). Column `WSPD`.
- `g`: peak gust (m/s). Column `GST`. If a gust is missing, we use the wind speed.

Then one row per hour: **average** pressure and wind over the hour that just ended, and the **maximum** gust in that hour. Gaps up to 2 hours are filled.

In [ ]:
# STEP 3: NDBC files -> pressure, wind, gust
def parse_ndbc(text):
    # Parses one NDBC 'stdmet' yearly file into a DataFrame with columns p, w, g.
    lines = [l for l in text.splitlines() if l.strip()]
    empty = pd.DataFrame(columns=["p", "w", "g"], dtype=float)
    if not lines:
        return empty
    header = lines[0].lstrip("#").split()
    rows = [l.split() for l in lines[1:] if not l.startswith("#")]
    rows = [r for r in rows if len(r) == len(header)]
    if not rows:
        return empty
    df = pd.DataFrame(rows, columns=range(len(header)))
    num = lambda name: (pd.to_numeric(df[header.index(name)], errors="coerce")
                        if name in header else pd.Series(np.nan, index=df.index))
    year = pd.to_numeric(df[0], errors="coerce")
    year = year.where(year >= 100, year + 1900)
    minute = pd.to_numeric(df[4], errors="coerce") if header[4] == "mm" else pd.Series(0, index=df.index)
    ts = pd.to_datetime(dict(year=year, month=pd.to_numeric(df[1], errors="coerce"),
                             day=pd.to_numeric(df[2], errors="coerce"),
                             hour=pd.to_numeric(df[3], errors="coerce"), minute=minute),
                        errors="coerce", utc=True)
    p = num("PRES") if "PRES" in header else num("BAR")
    w = num("WSPD")
    g = num("GST")
    p = p.where((p > P_MIN) & (p < P_MAX))
    w = w.where((w >= 0) & (w < W_MAX))
    g = g.where((g >= 0) & (g < G_MAX))
    g = g.fillna(w)                                   # missing gust -> use wind
    out = pd.DataFrame({"p": p.values, "w": w.values, "g": g.values}, index=ts)
    out = out[out.index.notna()].dropna(how="all").sort_index()
    return out[~out.index.duplicated()]

def download_station(stn):
    parts = []
    for yr in YEARS:
        path = f"data/ndbc/{stn}h{yr}.txt"
        if os.path.exists(path + ".missing"):
            continue
        if not os.path.exists(path):
            b = fetch(f"https://www.ndbc.noaa.gov/data/historical/stdmet/{stn}h{yr}.txt.gz")
            if b is None:
                open(path + ".missing", "w").close()
                continue
            with open(path, "w") as f:
                f.write(gzip.decompress(b).decode("latin-1"))
        with open(path) as f:
            d = parse_ndbc(f.read())
        if len(d):
            parts.append(d)
    if not parts:
        return pd.DataFrame(columns=["p", "w", "g"], dtype=float)
    d = pd.concat(parts).sort_index()
    return d[~d.index.duplicated()]

def to_hourly(raw):
    # Hour that just ended (label = end of hour): mean p, mean w, max g. No peeking ahead.
    r = raw.resample("1h", label="right", closed="right")
    h = pd.DataFrame({"p": r["p"].mean(), "w": r["w"].mean(), "g": r["g"].max()})
    full = pd.date_range(h.index.min(), h.index.max(), freq="1h")
    return h.reindex(full).interpolate(limit=2, limit_area="inside")

RAW = {}
for stn in STATION_POS:
    d = download_station(stn)
    if len(d):
        RAW[stn] = d
    print(f"{stn:6s} pressure {d['p'].notna().sum() if len(d) else 0:>9,}   wind {d['w'].notna().sum() if len(d) else 0:>9,}")
print(f"\n{len(RAW)} stations have data")

## Step 4: storm tracks (HURDAT2)

Same as version 1. If the download fails, download HURDAT2 from https://www.nhc.noaa.gov/data/#hurdat, upload it as `hurdat2.txt`, and re-run.

In [ ]:
# STEP 4: HURDAT2 storm tracks
TROPICAL = {"TS", "HU"}

def get_hurdat_text():
    if os.path.exists("hurdat2.txt"):
        return open("hurdat2.txt", encoding="latin-1").read()
    urls = []
    listing = fetch("https://www.nhc.noaa.gov/data/hurdat/")
    if listing:
        found = set(re.findall(r"hurdat2-1851-(\d{4})-(\d{6})\.txt", listing.decode("latin-1")))
        if found:
            y, d = max(found, key=lambda t: (int(t[0]), t[1][4:], t[1][:4]))
            urls.append(f"https://www.nhc.noaa.gov/data/hurdat/hurdat2-1851-{y}-{d}.txt")
    urls.append(HURDAT_FALLBACK)
    for u in urls:
        b = fetch(u)
        if b:
            print("HURDAT2:", u)
            text = b.decode("latin-1")
            with open("hurdat2.txt", "w", encoding="latin-1") as f:
                f.write(text)
            return text
    raise RuntimeError("Download HURDAT2 by hand and upload it as hurdat2.txt")

def parse_hurdat(text, min_year):
    rows, sid, name = [], None, None
    for line in text.splitlines():
        p = [x.strip() for x in line.split(",")]
        if len(p) < 3:
            continue
        if not p[0].isdigit():
            sid, name = p[0], p[1]
            continue
        if int(p[0][:4]) < min_year:
            continue
        lat = float(p[4][:-1]) * (1 if p[4][-1] == "N" else -1)
        lon = float(p[5][:-1]) * (1 if p[5][-1] == "E" else -1)
        rows.append((sid, name, p[0] + p[1].zfill(4), p[3], lat, lon))
    df = pd.DataFrame(rows, columns=["sid", "name", "ts", "status", "lat", "lon"])
    df["ts"] = pd.to_datetime(df.ts, format="%Y%m%d%H%M", utc=True)
    return df

def hourly_tracks(best):
    out = []
    for sid, g in best.groupby("sid"):
        g = g.drop_duplicates("ts").set_index("ts").sort_index()
        idx = pd.date_range(g.index.min().ceil("h"), g.index.max().floor("h"), freq="1h")
        if len(idx) == 0:
            continue
        gi = g.reindex(g.index.union(idx))
        gi[["lat", "lon"]] = gi[["lat", "lon"]].interpolate(method="time")
        gi["status"] = gi["status"].ffill()
        gi = gi.loc[idx]
        gi["sid"], gi["name"] = sid, g["name"].iloc[0]
        out.append(gi.rename_axis("ts").reset_index()[["sid", "name", "ts", "status", "lat", "lon"]])
    return pd.concat(out, ignore_index=True)

BEST = parse_hurdat(get_hurdat_text(), min_year=min(YEARS) - 1)
TRACKS = hourly_tracks(BEST)
print(f"{BEST.sid.nunique()} storms, {len(TRACKS):,} hourly track points")

## Step 5: labels + features

**Label** (same as version 1): 1 if a TS/HU center comes within 150 km in the next 12 hours.

**Pressure features** (from the barometer): `p_now`, `d1`, `d3`, `d6` (change over 1/3/6 h), `min6` (lowest in 6 h), `d3_rate` (is the drop speeding up?).

**Wind features** (from the anemometer):

| Feature | Meaning |
|---|---|
| `w_now` | average wind speed in the last hour (m/s) |
| `w_mean3` | average wind over the last 3 hours |
| `g_max6` | strongest gust in the last 6 hours |
| `dw3`, `dw6` | change in wind speed over 3 and 6 hours (is it picking up?) |

Every feature needs at most **7 hourly values** (now and the 6 hours before), so all three models warm up in about 6 hours.

In [ ]:
# STEP 5: labels, features, storm events
P_FEATURES = ["p_now", "d1", "d3", "d6", "min6", "d3_rate"]
W_FEATURES = ["w_now", "w_mean3", "g_max6", "dw3", "dw6"]
FEATURE_SETS = {"pressure": P_FEATURES, "wind": W_FEATURES, "both": P_FEATURES + W_FEATURES}
ALL_FEATURES = P_FEATURES + W_FEATURES

FEATURE_HELP = {
    "p_now": "pressure now", "d1": "1-hour pressure change", "d3": "3-hour pressure change",
    "d6": "6-hour pressure change", "min6": "lowest pressure in the last 6 hours",
    "d3_rate": "speed-up of the 3-hour pressure change",
    "w_now": "wind speed now", "w_mean3": "average wind over 3 hours",
    "g_max6": "strongest gust in the last 6 hours",
    "dw3": "3-hour change in wind speed", "dw6": "6-hour change in wind speed",
}
FEATURE_UNITS = {f: "hPa" for f in P_FEATURES} | {f: "m/s" for f in W_FEATURES}

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

HOURLY = {}   # hourly p/w/g per station, used later to check the Pi runtime

def build_station(stn, raw):
    lat, lon = STATION_POS[stn]
    df = to_hourly(raw)
    df.index.name = "time"
    HOURLY[stn] = df[["p", "w", "g"]].copy()
    # pressure features
    df["p_now"] = df.p
    df["d1"] = df.p.diff(1)
    df["d3"] = df.p.diff(3)
    df["d6"] = df.p.diff(6)
    df["min6"] = df.p.rolling(7).min()
    df["d3_rate"] = df.d3.diff(1)
    # wind features
    df["w_now"] = df.w
    df["w_mean3"] = df.w.rolling(3).mean()
    df["g_max6"] = df.g.rolling(7).max()
    df["dw3"] = df.w.diff(3)
    df["dw6"] = df.w.diff(6)

    tr = TRACKS.copy()
    tr["dist"] = haversine_km(lat, lon, tr.lat.values, tr.lon.values)
    near = tr[(tr.dist <= RADIUS_KM) & tr.status.isin(TROPICAL)]
    close = pd.Series(df.index.isin(near.ts).astype(float), index=df.index)
    df["label"] = close[::-1].rolling(HORIZON_H, min_periods=1).max()[::-1].shift(-1)
    wide = pd.Series(df.index.isin(tr[tr.dist <= NORMAL_KM].ts).astype(float), index=df.index)
    df["normal"] = wide.rolling(2 * 48 + 1, center=True, min_periods=1).max() == 0

    df["station"] = stn
    df["year"] = df.index.year
    df = df[df.index.month.isin(SEASON_MONTHS)].dropna(subset=["label"])
    df["label"] = df.label.astype(int)

    events = []
    for sid, g in near.groupby("sid"):
        r = g.loc[g.dist.idxmin()]
        events.append(dict(station=stn, sid=sid, name=r["name"], year=r.ts.year,
                           t_closest=r.ts, min_dist_km=round(float(r.dist), 1)))
    return df, events

frames, all_events = [], []
for stn, raw in RAW.items():
    df, ev = build_station(stn, raw)
    frames.append(df)
    all_events += ev
DATA = pd.concat(frames).reset_index().sort_values(["station", "time"]).reset_index(drop=True)
EVENTS = pd.DataFrame(all_events)

# Fair comparison: every model is trained and tested on the SAME hours (both pressure and wind present).
COMMON = DATA.dropna(subset=ALL_FEATURES).reset_index(drop=True)
print(f"{len(DATA):,} station-hours total; {len(COMMON):,} have both pressure and wind "
      f"({100*len(COMMON)/max(len(DATA),1):.0f}%)")
print(f"{COMMON.label.sum():,} 'storm coming' hours in the common set; {len(EVENTS)} storm events")

## Step 6: split by year, then train 6 models

For each feature set (`pressure`, `wind`, `both`) we train a **logistic regression** and a **decision tree (depth 3)**, all with `class_weight="balanced"` because storms are rare.

In [ ]:
# STEP 6: split + train
def part(years):
    return COMMON[COMMON.year.isin(years)].reset_index(drop=True)

TRAIN, VAL, TEST = part(TRAIN_YEARS), part(VAL_YEARS), part(TEST_YEARS)
for n, d in [("train", TRAIN), ("val", VAL), ("test", TEST)]:
    print(f"{n:5s} {len(d):>9,} hours  {d.label.sum():>6,} storm-coming hours")

def X(d, feats):
    return d[feats].to_numpy(dtype=float)

def new_model(kind):
    if kind == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))
    return DecisionTreeClassifier(max_depth=MAX_DEPTH, min_samples_leaf=MIN_LEAF,
                                  class_weight="balanced", random_state=0)

MODELS = {}
for set_name, feats in FEATURE_SETS.items():
    for kind in ["logreg", "tree"]:
        m = new_model(kind)
        m.fit(X(TRAIN, feats), TRAIN.label)
        MODELS[(set_name, kind)] = m
print("trained", len(MODELS), "models")

## Step 7: thresholds + fair comparison on unseen storms

Same rule as version 1, applied to every model:
1. **Threshold** on the validation years: lowest threshold with **≤ 1 false alarm per week** of normal weather.
2. **Test** on 2022+ storms: storms caught (alert in the 24 h before closest approach), lead time, false alarms.
3. **Within each set**, the tree wins if it is within 5 recall points of logistic regression.

In [ ]:
# STEP 7: thresholds + comparison
def false_alarms_per_week(df, alert):
    alert = np.asarray(alert, dtype=bool)
    same_run = ((df.station == df.station.shift(1)) &
                (df.time - df.time.shift(1) == pd.Timedelta("1h"))).to_numpy()
    prev = np.r_[False, alert[:-1]] & same_run
    onsets = alert & ~prev & df.normal.to_numpy()
    weeks = df.normal.sum() / 168
    return onsets.sum() / weeks if weeks else float("nan")

def event_metrics(df, alert, years):
    alert = np.asarray(alert, dtype=bool)
    by_stn = {s: (g.time.dt.tz_convert(None).to_numpy(), alert[g.index.to_numpy()])
              for s, g in df.groupby("station")}
    hour = np.timedelta64(1, "h")
    caught, leads, n = 0, [], 0
    for e in EVENTS[EVENTS.year.isin(years)].itertuples():
        if e.station not in by_stn:
            continue
        times, al = by_stn[e.station]
        tc = np.datetime64(e.t_closest.tz_convert(None))
        win = (times >= tc - 24 * hour) & (times <= tc)
        if win.sum() < 12:
            continue
        n += 1
        if al[win].any():
            caught += 1
            leads.append((tc - times[win][al[win]].min()) / hour)
    return dict(events=n, recall=round(100 * caught / n, 1) if n else float("nan"),
                median_lead_h=round(float(np.median(leads)), 1) if leads else 0.0)

def pick_threshold(model, df, feats):
    prob = model.predict_proba(X(df, feats))[:, 1]
    fa = float("nan")
    for thr in np.round(np.arange(0.05, 1.0, 0.01), 2):
        fa = false_alarms_per_week(df, prob >= thr)
        if fa <= FA_PER_WEEK_TARGET:
            return float(thr), fa
    return 0.99, fa

RESULTS = {}
for (set_name, kind), m in MODELS.items():
    feats = FEATURE_SETS[set_name]
    thr, fa_val = pick_threshold(m, VAL, feats)
    alert = m.predict_proba(X(TEST, feats))[:, 1] >= thr
    ev = event_metrics(TEST, alert, TEST_YEARS)
    RESULTS[f"{set_name}/{kind}"] = dict(
        set=set_name, model=kind, threshold=thr,
        storms_caught_pct=ev["recall"], events=ev["events"], median_lead_h=ev["median_lead_h"],
        false_alarms_per_week=round(false_alarms_per_week(TEST, alert), 2),
        false_alarms_per_week_val=round(fa_val, 2),
    )

COMPARE = pd.DataFrame(RESULTS).T
try:
    display(COMPARE)
except NameError:
    print(COMPARE)

WINNERS = {}
for set_name in FEATURE_SETS:
    lr = RESULTS[f"{set_name}/logreg"]["storms_caught_pct"]
    tr = RESULTS[f"{set_name}/tree"]["storms_caught_pct"]
    WINNERS[set_name] = "tree" if tr >= lr - 5 else "logreg"
    print(f"{set_name:8s} -> {WINNERS[set_name]:6s} (tree {tr}% vs logistic regression {lr}%)")

def score(set_name):
    r = RESULTS[f"{set_name}/{WINNERS[set_name]}"]
    return (r["storms_caught_pct"], r["median_lead_h"], -r["false_alarms_per_week"])   # catch more, then warn earlier
REPLAY_SET = max(["both", "pressure", "wind"], key=score)
print(f"\nLive (barometer down): {LIVE_SET}/{WINNERS[LIVE_SET]}")
print(f"Replay (has pressure + wind): {REPLAY_SET}/{WINNERS[REPLAY_SET]}")

## Step 8: pictures for the slides
- One tree picture per set (the dashboard draws the live one with its path highlighted)
- A comparison chart of the three sets
- One real test storm: pressure and wind on top, the three risk scores below

In [ ]:
# STEP 8: figures
for set_name, feats in FEATURE_SETS.items():
    tree = MODELS[(set_name, "tree")]
    print(f"--- {set_name} tree ---")
    print(export_text(tree, feature_names=feats, decimals=2))
    fig, ax = plt.subplots(figsize=(16, 7))
    plot_tree(tree, feature_names=feats, class_names=["no storm", "storm coming"],
              filled=True, rounded=True, proportion=True, impurity=False, ax=ax, fontsize=9)
    ax.set_title(f"StormBox storm risk tree: {set_name} (depth {MAX_DEPTH})")
    fig.savefig(f"out/figures/decision_tree_{set_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

labels = [f"{s}\n({WINNERS[s]})" for s in FEATURE_SETS]
colors = ["#7a8ca3", "#2f6db5", "#1b3f73"]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, col, title in zip(axes, ["storms_caught_pct", "median_lead_h", "false_alarms_per_week"],
                          ["Storms caught (%)", "Median lead time (h)", "False alarms / week"]):
    vals = [RESULTS[f"{s}/{WINNERS[s]}"][col] for s in FEATURE_SETS]
    ax.bar(labels, vals, color=colors)
    ax.set_title(title)
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v}", ha="center", va="bottom")
fig.suptitle("Test storms 2022+ (never seen in training), same hours for every model")
fig.tight_layout()
fig.savefig("out/figures/model_comparison_sets.png", dpi=150, bbox_inches="tight")
plt.show()

tev = EVENTS[EVENTS.year.isin(TEST_YEARS)].sort_values("min_dist_km")
for e in tev.itertuples():
    d = TEST[(TEST.station == e.station) &
             (TEST.time >= e.t_closest - pd.Timedelta("48h")) &
             (TEST.time <= e.t_closest + pd.Timedelta("24h"))]
    if len(d) < 48:
        continue
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
    a1.plot(d.time, d.p_now, color="black", label="pressure (hPa)")
    a1.set_ylabel("hPa")
    a1b = a1.twinx()
    a1b.plot(d.time, d.w_now, color="#c0392b", label="wind (m/s)")
    a1b.plot(d.time, d.g_max6, color="#e6a29a", ls="--", label="max gust 6 h (m/s)")
    a1b.set_ylabel("m/s")
    for s, c in zip(FEATURE_SETS, colors):
        m = MODELS[(s, WINNERS[s])]
        a2.plot(d.time, m.predict_proba(X(d, FEATURE_SETS[s]))[:, 1], color=c, label=f"{s} risk")
        a2.axhline(RESULTS[f"{s}/{WINNERS[s]}"]["threshold"], color=c, ls=":", lw=1)
    for a in (a1, a2):
        a.axvline(e.t_closest, color="red", ls="--")
    a2.set_ylim(0, 1.05)
    a2.set_ylabel("risk score")
    a1.set_title(f"{e.name.title()} {e.year} at station {e.station} ({e.min_dist_km} km away)")
    h1, l1 = a1.get_legend_handles_labels()
    h2, l2 = a1b.get_legend_handles_labels()
    a1.legend(h1 + h2, l1 + l2, loc="lower left", fontsize=8)
    a2.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    fig.savefig("out/figures/test_storm_example_sets.png", dpi=150, bbox_inches="tight")
    plt.show()
    break

## Step 9: export tiny models + a pure-Python runtime

- `storm_risk_pressure.json`, `storm_risk_wind.json`, `storm_risk_both.json`: one file per set (a few KB each), each with its tree, logistic regression, chosen model (`"use"`), and threshold.
- `storm_risk.json`: same as the pressure file, so older code keeps working.
- `storm_risk_runtime.py`: plain Python, no scikit-learn. New: `features_from_history(model, hist)` builds whatever features the model needs from hourly pressure and/or wind. The cell checks it gives exactly the same answers as the notebook.

**Feeding it on the box:** each hour, keep the **average** sea-level pressure, the **average** wind speed (m/s), and the **maximum** gust (m/s). Pass the last 7 of each, oldest first.

In [ ]:
# STEP 9: export JSON models + runtime
def tree_to_json(t, feats):
    T = t.tree_
    nodes = []
    for i in range(T.node_count):
        v = T.value[i][0]
        node = {"id": i, "samples": int(T.n_node_samples[i]),
                "risk": float(v[1] / v.sum()) if v.sum() else 0.0}
        if T.children_left[i] != -1:
            node.update(feature=feats[T.feature[i]], threshold=float(T.threshold[i]),
                        left=int(T.children_left[i]), right=int(T.children_right[i]))
        nodes.append(node)
    return {"nodes": nodes}

def logreg_to_json(pipe):
    sc, lr = pipe.named_steps["standardscaler"], pipe.named_steps["logisticregression"]
    return {"mean": sc.mean_.tolist(), "scale": sc.scale_.tolist(),
            "coef": lr.coef_[0].tolist(), "intercept": float(lr.intercept_[0])}

VERSION = "storm-risk-v2-" + time.strftime("%Y%m%d")
EXPORTED = {}
for set_name, feats in FEATURE_SETS.items():
    w = WINNERS[set_name]
    model = {
        "version": f"{VERSION}-{set_name}",
        "set": set_name,
        "use": w,
        "threshold": RESULTS[f"{set_name}/{w}"]["threshold"],
        "thresholds": {k: RESULTS[f"{set_name}/{k}"]["threshold"] for k in ["logreg", "tree"]},
        "features": feats,
        "feature_help": {f: FEATURE_HELP[f] for f in feats},
        "feature_units": {f: FEATURE_UNITS[f] for f in feats},
        "label": f"TS/HU center within {RADIUS_KM} km in the next {HORIZON_H} h",
        "inputs": "hourly: mean sea-level pressure (hPa), mean wind (m/s), max gust (m/s); last 7 hours, oldest first",
        "tree": tree_to_json(MODELS[(set_name, "tree")], feats),
        "logreg": logreg_to_json(MODELS[(set_name, "logreg")]),
    }
    EXPORTED[set_name] = model
    with open(f"out/models/storm_risk_{set_name}.json", "w") as f:
        json.dump(model, f, indent=1)
    joblib.dump(MODELS[(set_name, "tree")], f"out/models/storm_risk_{set_name}_tree.joblib")
    joblib.dump(MODELS[(set_name, "logreg")], f"out/models/storm_risk_{set_name}_logreg.joblib")
with open("out/models/storm_risk.json", "w") as f:          # backward compatible name
    json.dump(EXPORTED["pressure"], f, indent=1)

METRICS = {
    "version": VERSION, "results_test": RESULTS, "winners": WINNERS,
    "recommended": {"live_while_barometer_down": LIVE_SET, "replay": REPLAY_SET, "live_if_barometer_works": "pressure"},
    "rule": "lowest threshold with <= 1 false alarm/week on validation years; tree wins within a set if within 5 recall points",
    "comparison": "all models trained and tested on the same station-hours (pressure and wind both present)",
    "train_years": [min(TRAIN_YEARS), max(TRAIN_YEARS)], "val_years": [min(VAL_YEARS), max(VAL_YEARS)],
    "test_years": [min(TEST_YEARS), max(TEST_YEARS)],
    "stations": sorted(RAW), "radius_km": RADIUS_KM, "horizon_h": HORIZON_H,
    "test_events": EVENTS[EVENTS.year.isin(TEST_YEARS)].assign(t_closest=lambda d: d.t_closest.astype(str)).to_dict("records"),
    "tree_text": {s: export_text(MODELS[(s, "tree")], feature_names=f, decimals=2) for s, f in FEATURE_SETS.items()},
}
with open("out/models/metrics.json", "w") as f:
    json.dump(METRICS, f, indent=1, default=str)

RUNTIME = """\"\"\"StormBox storm-risk runtime v2: plain Python, no scikit-learn needed.

    model = load("ai/ml/storm_risk_wind.json")          # or _pressure / _both
    hist = {"p": [...7 hourly sea-level hPa...],          # only what the model needs
            "w": [...7 hourly mean wind m/s...],
            "g": [...7 hourly max gust m/s...]}          # g optional (falls back to w)
    feats = features_from_history(model, hist)          # oldest first: [t-6h, ..., now]
    result = predict(model, feats)                      # {"risk": 0.78, "alert": True, "path": [...]}
    explain(model, result)                              # "Storm risk 78% because ..."
\"\"\"
import json, math

P_FEATURES = ["p_now", "d1", "d3", "d6", "min6", "d3_rate"]
W_FEATURES = ["w_now", "w_mean3", "g_max6", "dw3", "dw6"]

def load(path="ai/ml/storm_risk_wind.json"):
    with open(path) as f:
        return json.load(f)

def needs(model):
    \"\"\"Which inputs the model needs: a set like {"p"}, {"w"} or {"p", "w"}.\"\"\"
    f = set(model["features"])
    return ({"p"} if f & set(P_FEATURES) else set()) | ({"w"} if f & set(W_FEATURES) else set())

def to_sea_level(p_hpa, altitude_m, temp_c=25.0):
    \"\"\"Barometer reading at the box -> sea-level pressure (what the model was trained on).\"\"\"
    if not altitude_m:
        return p_hpa
    return p_hpa * (1 - 0.0065 * altitude_m / (temp_c + 0.0065 * altitude_m + 273.15)) ** -5.257

def _last7(values, name):
    v = list(values)[-7:]
    if len(v) < 7 or any(x is None for x in v):
        raise ValueError(f"need 7 hourly {name} values (t-6h ... now)")
    return [float(x) for x in v]

def features_from_hourly(p):
    \"\"\"Pressure features from 7 hourly sea-level pressures, oldest first (kept for older code).\"\"\"
    p = _last7(p, "pressure")
    now = p[-1]
    d3 = now - p[-4]
    return {"p_now": now, "d1": now - p[-2], "d3": d3, "d6": now - p[0],
            "min6": min(p), "d3_rate": d3 - (p[-2] - p[-5])}

def wind_features(w, g=None):
    \"\"\"Wind features from 7 hourly mean winds and 7 hourly max gusts (m/s), oldest first.\"\"\"
    w = _last7(w, "wind")
    g = _last7(g, "gust") if g is not None else w
    return {"w_now": w[-1], "w_mean3": sum(w[-3:]) / 3, "g_max6": max(g),
            "dw3": w[-1] - w[-4], "dw6": w[-1] - w[0]}

def features_from_history(model, hist):
    feats = {}
    if "p" in needs(model):
        feats.update(features_from_hourly(hist["p"]))
    if "w" in needs(model):
        feats.update(wind_features(hist["w"], hist.get("g")))
    return feats

def predict(model, feats):
    return _tree(model, feats) if model["use"] == "tree" else _logreg(model, feats)

def _tree(model, feats):
    nodes, i, path = model["tree"]["nodes"], 0, []
    while "feature" in nodes[i]:
        n = nodes[i]
        x = feats[n["feature"]]
        left = x <= n["threshold"]
        path.append({"node": i, "feature": n["feature"], "value": x,
                     "threshold": n["threshold"], "op": "<=" if left else ">"})
        i = n["left"] if left else n["right"]
    risk = nodes[i]["risk"]
    return {"model": "tree", "set": model.get("set"), "risk": risk,
            "alert": risk >= model["threshold"], "path": path, "leaf": i}

def _logreg(model, feats):
    m = model["logreg"]
    z, parts = m["intercept"], []
    for f, mu, sd, w in zip(model["features"], m["mean"], m["scale"], m["coef"]):
        c = w * (feats[f] - mu) / sd
        z += c
        parts.append((f, c))
    z = max(-30.0, min(30.0, z))
    risk = 1 / (1 + math.exp(-z))
    return {"model": "logreg", "set": model.get("set"), "risk": risk,
            "alert": risk >= model["threshold"],
            "top_factors": sorted(parts, key=lambda t: -abs(t[1]))[:3]}

def _fmt(model, f, v):
    unit = model.get("feature_units", {}).get(f, "hPa")
    if unit == "m/s":
        return f"{v:.1f} m/s (about {abs(v) * 2.237:.0f} mph)"
    return f"{v:.1f} hPa"

def explain(model, result):
    \"\"\"One plain-English sentence the agent can say or the dashboard can show.\"\"\"
    h = model["feature_help"]
    pct = round(100 * result["risk"])
    if "path" in result:
        last = {}
        for s in result["path"]:
            last[s["feature"]] = s
        why = [f"{h[f]} is {_fmt(model, f, s['value'])} "
               f"({'at or below' if s['op'] == '<=' else 'above'} {s['threshold']:.1f})"
               for f, s in last.items()]
    else:
        why = [f"{h[f]} {'raises' if c > 0 else 'lowers'} the risk" for f, c in result["top_factors"]]
    return f"Storm risk score {pct} because " + "; ".join(why) + "."
"""
with open("out/models/storm_risk_runtime.py", "w") as f:
    f.write(RUNTIME)

# Check 1: runtime predictions == scikit-learn, for every set and model type
rt = {}
exec(RUNTIME, rt)
sample = TEST.sample(min(3000, len(TEST)), random_state=0)
for set_name, feats in FEATURE_SETS.items():
    for kind in ["tree", "logreg"]:
        mdl = dict(EXPORTED[set_name], use=kind)
        ours = np.array([rt["predict"](mdl, r)["risk"] for r in sample[feats].to_dict("records")])
        theirs = MODELS[(set_name, kind)].predict_proba(X(sample, feats))[:, 1]
        assert np.allclose(ours, theirs, atol=1e-6), (set_name, kind)
print("Check 1 passed: runtime predictions match scikit-learn for all 6 models")

# Check 2: runtime features from raw hourly history == notebook features
checked = 0
for r in sample.head(300).itertuples():
    win = HOURLY[r.station].loc[r.time - pd.Timedelta("6h"): r.time]
    if len(win) != 7 or win.isna().any().any():
        continue
    hist = {"p": win.p.tolist(), "w": win.w.tolist(), "g": win.g.tolist()}
    feats = rt["features_from_history"](EXPORTED["both"], hist)
    for f in ALL_FEATURES:
        assert abs(feats[f] - getattr(r, f)) < 1e-6, (f, feats[f], getattr(r, f))
    checked += 1
print(f"Check 2 passed: runtime features match the notebook on {checked} hours")

for s in FEATURE_SETS:
    top = sample.assign(r=MODELS[(s, WINNERS[s])].predict_proba(X(sample, FEATURE_SETS[s]))[:, 1]).sort_values("r").iloc[-1]
    print(f"{s:8s} example:", rt["explain"](EXPORTED[s], rt["predict"](EXPORTED[s], top[FEATURE_SETS[s]].to_dict())))

## Step 10: real storms for the demo replay (now with wind)

Each file: `ts,pressure_hpa,wind_ms,gust_ms`, one row every 10 minutes, 36 h before to 18 h after closest approach. `pressure_hpa` stays the second column, so code that reads version-1 replay files still works. Storms whose station has good wind data are picked first.

In [ ]:
# STEP 10: replay CSVs
def export_replays(max_storms=6, before_h=36, after_h=18):
    index, done = [], set()
    tev = EVENTS[EVENTS.year.isin(TEST_YEARS)].sort_values("min_dist_km")
    candidates = []
    for e in tev.itertuples():
        if e.station not in RAW:
            continue
        start, end = e.t_closest - pd.Timedelta(hours=before_h), e.t_closest + pd.Timedelta(hours=after_h)
        seg = RAW[e.station][start - pd.Timedelta("1h"): end + pd.Timedelta("1h")]
        grid = pd.date_range(start.ceil("10min"), end.floor("10min"), freq="10min")
        if len(seg) == 0:
            continue
        s = seg.reindex(seg.index.union(grid)).interpolate(method="time", limit=6, limit_area="inside").reindex(grid)
        p_ok, w_ok = s.p.notna().mean() >= 0.9, s.w.notna().mean() >= 0.9
        if not p_ok:
            continue
        candidates.append((0 if w_ok else 1, e.min_dist_km, e, s, w_ok))
    for _, _, e, s, w_ok in sorted(candidates, key=lambda c: (c[0], c[1])):
        if e.sid in done or len(done) >= max_storms:
            continue
        s = s.interpolate(limit_direction="both").round(1)
        out = pd.DataFrame({"ts": s.index.strftime("%Y-%m-%dT%H:%M:%SZ"), "pressure_hpa": s.p.values})
        if w_ok:
            out["wind_ms"] = s.w.values
            out["gust_ms"] = s.g.values
        fname = f"{e.name.lower()}_{e.year}_{e.station}.csv"
        out.to_csv(f"out/replay/{fname}", index=False)
        done.add(e.sid)
        index.append(dict(file=fname, storm=e.name.title(), year=int(e.year), station=e.station,
                          closest_km=e.min_dist_km, closest_time=str(e.t_closest), has_wind=bool(w_ok),
                          min_pressure_hpa=float(s.p.min()),
                          max_gust_ms=float(s.g.max()) if w_ok else None, rows=len(s)))
        print(f"{fname:32s} min {s.p.min():.1f} hPa" + (f", max gust {s.g.max():.1f} m/s" if w_ok else ", no wind")
              + f", {e.min_dist_km} km away")
    with open("out/replay/index.json", "w") as f:
        json.dump(index, f, indent=1)
    return index

REPLAYS = export_replays()

## Step 11: download everything

| File | Goes to |
|---|---|
| `models/storm_risk_wind.json`, `_pressure.json`, `_both.json`, `storm_risk.json` | `ai/ml/` |
| `models/storm_risk_runtime.py` | `ai/ml/` (replaces version 1; still has `features_from_hourly`) |
| `models/metrics.json` | `ai/ml/` |
| `models/*.joblib` | `ai/ml/backup/` |
| `replay/*.csv`, `replay/index.json` | `ai/ml/replay/` (replaces version 1 files) |
| `figures/*.png` | `ai/ml/figures/` |

In [ ]:
# STEP 11: zip + download
shutil.make_archive("stormbox_storm_model_v2", "zip", "out")
print("Created stormbox_storm_model_v2.zip")
try:
    from google.colab import files
    files.download("stormbox_storm_model_v2.zip")
except ImportError:
    print("Not running in Colab: the zip is next to this notebook.")

## What to try if the results look bad

| Problem | Try |
|---|---|
| Wind model catches few storms | `MAX_DEPTH = 4`, or `HORIZON_H = 18` (wind picks up later than pressure drops) |
| Too many false alarms | Lower `FA_PER_WEEK_TARGET` (e.g. 0.5) |
| Few hours in the common set | Some stations have no wind; that's fine. Add more stations if test events < 10 |

The comparison table + `model_comparison_sets.png` is the new ML slide: **"pressure is the best early-warning signal; wind still works when the barometer doesn't."**